
# 練習問題: 複数スレッドで挨拶する

## 目標

`#pragma omp parallel` (Fortran では `!$omp parallel`) を1つ挿入するだけで, 1つの `printf` (`print`) 文を複数のスレッドに実行させられることを体験する.

## 課題

`hello_threads.cpp` (または `hello_threads.f90`) は, 現状では1スレッドで1行だけ表示する.
コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, 挨拶がスレッドの数だけ表示されるようにせよ.

- C++: 挨拶を表示するブロック `{ ... }` の直前に `#pragma omp parallel` を1行加える.
- Fortran: `print` 文を `!$omp parallel` と `!$omp end parallel` で囲む.

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore hello_threads.cpp -o hello_threads.exe

# Fortran
nvfortran -fast -mp=multicore hello_threads.f90 -o hello_threads.exe
```

```
OMP_NUM_THREADS=4 ./hello_threads.exe
```

## 期待される結果

`OMP_NUM_THREADS` に指定した数だけ挨拶が表示される (順番は実行ごとに入れ替わってよい). 例:

```
hello from thread 0 of 4
hello from thread 2 of 4
hello from thread 1 of 4
hello from thread 3 of 4
```

`OMP_NUM_THREADS` の値を変えて, 表示される行数が変わることを確認せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ hello_threads.cpp
#include <cstdio>
#include <omp.h>

int main() {
  // TODO: 下のブロックの直前に #pragma omp parallel を1行追加し, printf を複数のスレッドで実行させよ.
  {
    printf("hello from thread %d of %d\n",
           omp_get_thread_num(), omp_get_num_threads());
  }
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore hello_threads.cpp -o hello_threads_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./hello_threads_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ hello_threads.f90
program hello_threads
  use omp_lib
  ! TODO: 下の print を !$omp parallel ... !$omp end parallel で囲み, 複数のスレッドで実行させよ.
  print "(a,i0,a,i0)", "hello from thread ", omp_get_thread_num(), &
       " of ", omp_get_num_threads()
  ! TODO: 上で始めた parallel 領域を閉じる (!$omp end parallel).
end program hello_threads

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore hello_threads.f90 -o hello_threads_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./hello_threads_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:hello_threads.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:hello_threads.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:hello_threads.cpp}